# Step 3 — Where is the eye actually looking? (gaze + head features)

EAR tells us *how open* an eye is, not *where it's pointed*. This notebook
adds that: for every frame, where does the iris sit relative to the rest
of the eye, and what is the head doing (turned, tilted, near or far)?

These features feed two different things later:
- **`step_05`** uses the head/gaze features as *extra context* for the
  blink classifier (the hope: EAR alone confuses "looking down" with
  "closed"; gaze context might not).
- **`step_06`** uses them as the actual input for a gaze-*direction*
  classifier.


## At a glance

| | |
|---|---|
| **Workshop step** | 3 of 6 |
| **Needs in this folder** | all your clips: `blinks_<name>.mp4`, `eyes_only_<name>.mp4`, `head_only_<name>.mp4` (see `README.md`). In a group, everyone's. |
| **Produces** | one `features_<clip>.csv` per clip, for example `features_eyes_only_<name>.csv` (one row of gaze and head features per frame). Steps 4, 5 and 6 read these. |
| **Run time** | roughly 10 to 20 seconds per clip on a normal laptop |
| **Next** | `step_04_label_intervals.py` (run from the terminal, see `README.md`) |

**How to run:** in the menu choose *Run > Restart Kernel and Run All Cells*, or press Shift+Enter on each cell from the top down. Do not skip cells: later cells use variables made by earlier ones.

MediaPipe prints warnings in red or pink (libEGL, TensorFlow Lite, "feedback manager", protobuf). They are harmless, ignore them.

## Iris offset: a coordinate system per eye

For each eye we define a small local coordinate system from its two
corners:
- an **axis** running corner-to-corner (the eye's "width" direction)
- a **normal** perpendicular to that (the eye's "height" direction)

Then we measure the iris centre's position relative to the eye's
midpoint, **in units of eye width** — not pixels, so it doesn't matter how
close someone is to the camera:
- `x`: along the eye axis → positive means toward the right of the image
- `y`: across the axis → positive means downward in the image

**Left and right in this workshop always mean left and right as seen in
the video.** So a positive `gaze_x` is a look towards the right edge of
the picture. The video is not mirrored, which makes that the person's own
*left*. The eye names (`r_x`, `l_x`, "right eye") are the exception: they
say which of the person's eyes a number comes from, not which way it is
looking.

This is more involved than "iris pixel x minus eye-corner pixel x" because
it stays correct even if the head is tilted — the eye's own axis rotates
with it.


In [ ]:
%matplotlib inline
import glob
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import matplotlib.pyplot as plt

R_EYE = [33, 160, 158, 133, 153, 144]
L_EYE = [362, 385, 387, 263, 373, 380]
R_IRIS, L_IRIS, NOSE = 468, 473, 1


def eye_aspect_ratio(p):
    v = np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])
    return v / (2.0 * np.linalg.norm(p[0] - p[3]))


def iris_offset(pts, a, b, iris):
    """Iris center relative to the eye center, in units of eye width.
    x: along the eye axis (+ = toward the right of the image, which is
       what "right" means everywhere in this workshop)
    y: across it (+ = down in the image)."""
    pa, pb = pts[a], pts[b]
    axis = pb - pa
    width = np.linalg.norm(axis)
    u = axis / width
    n = np.array([-u[1], u[0]])
    d = pts[iris] - (pa + pb) / 2
    return d @ u / width, d @ n / width


## Head features

We also track what the *head* is doing, separately from the eyes:
- `nose_dx`, `nose_dy`: nose tip position relative to the midpoint between
  the eyes, scaled by inter-eye distance (so it's a ratio, not pixels) — a
  rough stand-in for head turn (dx) and head nod (dy).
- `roll_deg`: tilt of the line between the eyes.
- `face_cx`, `face_cy`, `face_scale`: where the face sits in frame and how
  big it is (closer = larger).

Why bother with any of this if we only care about the eyes? Because head
pose and gaze direction get tangled together in a 2D video — see the
discussion at the end of this notebook.


In [ ]:
def extract_features(video_path):
    """Run face mesh over a whole clip, return one row of features per frame."""
    mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1, refine_landmarks=True,
        min_detection_confidence=0.5, min_tracking_confidence=0.5,
    )
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise SystemExit(f"Could not open {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    rows = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        t = len(rows) / fps
        result = mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if not result.multi_face_landmarks:
            rows.append({"time_s": t})   # no face: every feature is left empty (NaN)
            continue

        lm = result.multi_face_landmarks[0].landmark
        pts = np.array([[p.x * w, p.y * h] for p in lm])

        rx, ry = iris_offset(pts, 33, 133, R_IRIS)
        lx, ly = iris_offset(pts, 362, 263, L_IRIS)

        eye_mid = (pts[33] + pts[263]) / 2
        eye_vec = pts[263] - pts[33]
        scale = np.linalg.norm(eye_vec)
        rows.append({
            "time_s": t,
            "ear": (eye_aspect_ratio(pts[R_EYE]) + eye_aspect_ratio(pts[L_EYE])) / 2,
            "r_x": rx, "r_y": ry, "l_x": lx, "l_y": ly,
            # TRY IT: gaze_x / gaze_y average the two eyes. The per-eye values
            # (r_x, l_x, ...) are saved too, so later steps can use those instead.
            "gaze_x": (rx + lx) / 2, "gaze_y": (ry + ly) / 2,
            "nose_dx": (pts[NOSE][0] - eye_mid[0]) / scale,
            "nose_dy": (pts[NOSE][1] - eye_mid[1]) / scale,
            "roll_deg": np.degrees(np.arctan2(eye_vec[1], eye_vec[0])),
            "face_cx": eye_mid[0] / w, "face_cy": eye_mid[1] / h,
            "face_scale": scale / w,
        })
    cap.release()
    mesh.close()
    return pd.DataFrame(rows)


**Before running this:** in an `eyes_only` clip the head stays still while
the eyes move around. What should `nose_dx`/`nose_dy` look like across the
whole clip if the head genuinely doesn't move? (Perfectly flat at zero, or
something else?)

In [ ]:
EYES_CLIPS = sorted(glob.glob("eyes_only_*.mp4"))
if not EYES_CLIPS:
    raise SystemExit("No eyes_only clips found. Save your recording here as eyes_only_<name>.mp4 (see README.md).")
print("eyes_only clips:", EYES_CLIPS)

# We look closely at one clip first. The head stays still in an eyes_only
# clip, so it is the cleanest place to see pure eye movement.
# TRY IT: with several people's clips in the folder, change the 0 to 1 or 2
# to plot someone else, or put a file name here, e.g. "head_only_<name>.mp4".
# How different do two people's plots look for the same recording script?
VIDEO = EYES_CLIPS[0]

df = extract_features(VIDEO)
stem = os.path.splitext(VIDEO)[0]
df.to_csv(f"features_{stem}.csv", index=False)
print(f"Processed {len(df)} frames, saved features_{stem}.csv")
df.head()

### Hide blink frames, then smooth

A short rolling median (5 frames to start with) removes single-frame jitter without blurring real eye movements much.

In [ ]:
# ---------------------------------------------------------------------
# TRY IT: two numbers control how the plot below is cleaned up.
#   BLINK_EAR_FRACTION  a frame counts as "eyes closed" when its EAR is
#                       below this fraction of the clip's median EAR.
#                       Try 0.4 (hides less) and 0.8 (hides more).
#   SMOOTH_FRAMES       width of the rolling median, in frames. Try 1 (no
#                       smoothing) and 15. When does smoothing start to
#                       blur a real eye movement instead of removing noise?
# ---------------------------------------------------------------------
BLINK_EAR_FRACTION = 0.6
SMOOTH_FRAMES = 5

# the iris is partly hidden by the lids during a blink -- its measured
# position there is noise, not signal. Mask those frames out before
# looking at gaze.
open_eyes = df["ear"] >= BLINK_EAR_FRACTION * df["ear"].median()
plot_df = df.copy()
gaze_cols = ["r_x", "r_y", "l_x", "l_y", "gaze_x", "gaze_y"]
plot_df.loc[~open_eyes, gaze_cols] = np.nan
print(f"Frames with open eyes: {open_eyes.sum()} of {len(df)}")
smooth = plot_df.rolling(SMOOTH_FRAMES, center=True, min_periods=1).median()

### Plot the features over time

Top: horizontal iris offset (each eye, plus the average). Middle: vertical iris offset. Bottom: the two head features.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(smooth.time_s, smooth.r_x, lw=0.8, alpha=0.6, label="right eye")
ax[0].plot(smooth.time_s, smooth.l_x, lw=0.8, alpha=0.6, label="left eye")
ax[0].plot(smooth.time_s, smooth.gaze_x, lw=1.8, color="k", label="average")
ax[0].set_ylabel("iris x offset\n(eye widths)")
ax[0].legend(loc="upper right")
ax[1].plot(smooth.time_s, smooth.gaze_y, lw=1.5, color="k")
ax[1].set_ylabel("iris y offset\n(eye widths)")
ax[2].plot(smooth.time_s, smooth.nose_dx, label="nose dx (head turn)")
ax[2].plot(smooth.time_s, smooth.nose_dy, label="nose dy (head nod)")
ax[2].set_ylabel("head features")
ax[2].set_xlabel("time (s)")
ax[2].legend(loc="upper right")
fig.suptitle(stem)
plt.tight_layout()
plt.show()


## Discuss

- Look at the bottom panel (`nose_dx`/`nose_dy`). You predicted these
  would be flat for an `eyes_only` clip. Are they? If there's drift or noise even
  though the head "shouldn't" be moving, what does that tell you about how
  much to trust these features on their own?
- Look at the top panel: do the right and left eye (thin lines) ever
  disagree noticeably from their average (thick line)? When would
  averaging them hide something important?
- Notice the y-offset panel's resting level — is it sitting exactly at 0,
  or offset? (This comes up again in `step_06` and in
  `STUDENT_TASKS.md` task 3.)

## All the other clips

`step_05` (the blink classifier) needs these same features for every
`blinks` clip, and `step_06` needs them for every `eyes_only` clip and
every `head_only` clip (its control clips). Try calling
`extract_features(...)` yourself on one of your other clips and saving
the result as `features_<clip name>.csv` *before* running the cell below.
It is the same function, just applied to another clip, and a good check
that you actually understand what it's doing rather than just that it ran
once.


In [ ]:
# same function, every other clip in the folder: blinks clips for step_05,
# eyes_only and head_only clips for step_06
ALL_CLIPS = sorted(glob.glob("blinks_*.mp4") + glob.glob("eyes_only_*.mp4") + glob.glob("head_only_*.mp4"))
for clip in ALL_CLIPS:
    if clip == VIDEO:
        continue   # already done above
    clip_stem = os.path.splitext(clip)[0]
    df_clip = extract_features(clip)
    df_clip.to_csv(f"features_{clip_stem}.csv", index=False)
    missed = df_clip["ear"].isna().sum() if "ear" in df_clip else len(df_clip)
    print(f"Processed {len(df_clip)} frames ({missed} without a face), saved features_{clip_stem}.csv")

**Next:** `step_04_label_intervals.py` (a script with its own window, not a
notebook) is where a person hand-labels the footage. After that, `step_05`
and `step_06` use these feature files to see whether a
trained model can beat a simple threshold rule — first for blinks, then
for gaze direction.
